# Notebook: Real Estate Property Price Prediction
Romain Sebire - 125 009 460

**Objective:** Predict real estate property prices based on physical characteristics, location, and amenities, as part of a Kaggle in-class competition.

**Dataset:** 4,683 training samples and 2,000 test samples with 21 features including property type, neighborhood, area, number of rooms, and available amenities.

**Approach:** Feature extraction from text descriptions, feature engineering (total area, area per room), outlier treatment with IQR, and hyperparameter optimization with Optuna.

**Key Result:** Achieved a Kaggle score of **0.2527** (RMSPE) using LightGBM with Optuna-optimized hyperparameters.

## Table of Contents

1. [Library and Data Import](#Library-and-Data-Import)
2. [Exploratory Data Analysis](#Exploratory-Data-Analysis)
3. [Processing the Amenities Column](#Processing-the-'diferenciais'-(Features)-Column)
4. [Feature Engineering](#Feature-Engineering)
5. [Outlier Treatment](#Outlier-Treatment)
6. [Feature Importance](#Feature-Importance-with-LGBMClassifier)
7. [Data Preprocessing](#Data-Preprocessing)
8. [Model Training and Cross-Validation](#Model-Training-and-Cross-Validation)
9. [Hyperparameter Optimization (Optuna)](#Optimized-Hyperparameter-Search)
10. [Prediction and Submission](#Prediction-on-the-Test-Set-and-Submission-File-Preparation)
11. [Conclusion](#Conclusion)

## Feature Glossary

Since this dataset originates from a Brazilian Kaggle competition, all feature names are in Portuguese. Below is the translation:

| Portuguese | English | Type |
|------------|---------|------|
| `preco` | **price** (target) | numerical |
| `tipo` | property type | categorical |
| `bairro` | neighborhood | categorical |
| `tipo_vendedor` | seller type | categorical |
| `quartos` | bedrooms | numerical |
| `suites` | suites (en-suite bedrooms) | numerical |
| `vagas` | parking spaces | numerical |
| `area_util` | usable area (m²) | numerical |
| `area_extra` | extra area (m²) | numerical |
| `churrasqueira` | barbecue area | binary |
| `estacionamento` | parking | binary |
| `piscina` | swimming pool | binary |
| `playground` | playground | binary |
| `quadra` | sports court | binary |
| `s_festas` | party room | binary |
| `s_jogos` | game room | binary |
| `s_ginastica` | gym | binary |
| `sauna` | sauna | binary |
| `vista_mar` | ocean view | binary |
| `diferenciais` | amenities/features (free text) | text |

**Engineered features:** `area_total` (usable + extra area), `area_por_quarto` (area per bedroom), `nb_equipements` (count of amenities)

## Library and Data Import

In this cell, we import the essential libraries for data analysis, visualization, preprocessing, training, and evaluation of machine learning models.

In [ ]:
# --- Initial imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from collections import Counter
import optuna

In [ ]:
# --- Load data ---
train_df = pd.read_csv('data/conjunto_de_treinamento.csv')
test_df = pd.read_csv('data/conjunto_de_teste.csv')
sample_submission = pd.read_csv('data/exemplo_arquivo_respostas.csv')

print("Dimensões do conjunto de treinamento:", train_df.shape)
print("Dimensões do conjunto de teste:", test_df.shape)

The data was imported correctly: there are 4,683 rows in the training set and 2,000 rows in the test set.
There are 21 columns in the training set and only 20 in the test set, as the target column is absent.

## Exploratory Data Analysis

### Feature Display and Missing Value Count

In [ ]:
# Display all columns
pd.set_option('display.max_columns', None)

# Exibição das primeiras linhas
train_df.head()

In [ ]:
# Display feature list and types
train_df.info()

In [ ]:
# Display features with missing values (train_df)
missing_train = train_df.isnull().sum()
missing_train[missing_train > 0].sort_values(ascending=False)

# Display features with missing values (test_df)
missing_test = test_df.isnull().sum()
missing_test[missing_test > 0].sort_values(ascending=False)

No missing values in either the training or the test data.

### Distribution Plots of Categorical Features

Let's analyze the distribution of each feature. This allows us to visually explore the data distribution within each feature, which is useful for understanding its structure, detecting imbalances or anomalies, and guiding data treatment decisions.

In [ ]:
# List of categorical features for barplot display
features_categoricas = [
    'tipo', 
    'bairro', 
    'tipo_vendedor',
    'quartos', 
    'suites', 
    'vagas', 
#   'diferenciais',
    'churrasqueira', 
    'estacionamento',
    'piscina', 
    'playground', 
    'quadra',
    's_festas', 
    's_jogos', 
    's_ginastica',
    'sauna', 
    'vista_mar', 
]

# Loop through each column in the list
for col in features_categoricas:
    missing = train_df[col].isna().sum()
    print(f"\nDistribution of '{col}' (valores ausentes: {missing})")

    # Contagem das ocorrências (ignorando os NaN)
    counts = train_df[col].value_counts(dropna=True).sort_index()

    # Criação de um DataFrame para exibição
    data = pd.DataFrame({
        'value': counts.index.astype(str),
        'occurrences': counts.values
    })

    # Display the chart
    plt.figure(figsize=(12, 4))
    sns.barplot(data=data, x='value', y='occurrences', hue='value', palette='viridis', legend=False)
    plt.title(f"Distribution of '{col}' (NaN: {missing})")
    plt.xlabel(col)
    plt.ylabel("Number of Occurrences")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


### Distribution Plots of Numerical Features

In [ ]:
# Basic statistics
data = train_df['area_util']
min_val = data.min()
max_val = data.max()
mean_val = data.mean()
median_val = data.median()
q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
nan_count = data.isna().sum()

# Display the chart
plt.figure(figsize=(8, 5))
sns.histplot(data=data, kde=True, bins=300, color='skyblue')
plt.title("Distribution of 'area_util'")
plt.xlabel('area_util')
plt.ylabel("Frequency")
plt.xlim(0, upper_bound)

 # Linhas de estatísticas
plt.axvline(min_val, color='red', linestyle='--', label=f'Mín: {min_val:.2f}')
plt.axvline(max_val, color='green', linestyle='--', label=f'Máx: {max_val:.2f}')
plt.axvline(mean_val, color='orange', linestyle='--', label=f'Média: {mean_val:.2f}')
plt.axvline(median_val, color='purple', linestyle='--', label=f'Mediana: {median_val:.2f}')
plt.axvline(q1, color='blue', linestyle='--', label=f'Q1: {q1:.2f}')
plt.axvline(q3, color='blue', linestyle=':', label=f'Q3: {q3:.2f}')

 # Legenda
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Boxplot horizontal
plt.figure(figsize=(10, 1.5))
sns.boxplot(data=data, orient='h', color='lightcoral')
plt.title("Boxplot de 'area_util'")
plt.xlabel("area_util")
plt.tight_layout()
plt.show()

In [ ]:
# Original data including zeros
data = train_df['area_extra']
zeros_count = (data == 0).sum()
data = data[data != 0]

# Filter values above Q3 for the chart (keeping zeros)
q3 = data.quantile(0.75)
data_filtrada = data[data <= q3]

min_val = data_filtrada.min()
max_val = data_filtrada.max()
mean_val = data_filtrada.mean()
median_val = data_filtrada.median()
q1 = data_filtrada.quantile(0.25)

# Display the chart
plt.figure(figsize=(8, 5))
sns.histplot(data=data_filtrada, kde=True, bins=30, color='skyblue')
plt.title("Distribution of 'area_extra' (valores <= Q3)")
plt.xlabel('area_extra')
plt.ylabel("Frequency")

# Linhas de estatísticas
plt.axvline(min_val, color='red', linestyle='--', label=f'Mín: {min_val:.2f}')
plt.axvline(max_val, color='green', linestyle='--', label=f'Máx: {max_val:.2f}')
plt.axvline(mean_val, color='orange', linestyle='--', label=f'Média: {mean_val:.2f}')
plt.axvline(median_val, color='purple', linestyle='--', label=f'Mediana: {median_val:.2f}')
plt.axvline(q1, color='blue', linestyle='--', label=f'Q1: {q1:.2f}')
plt.axvline(q3, color='blue', linestyle=':', label=f'Q3: {q3:.2f}')

# Legenda incluindo a contagem de zeros
plt.legend(title=f'Zeros: {zeros_count}', loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Boxplot horizontal
plt.figure(figsize=(10, 1.5))
sns.boxplot(data=data_filtrada, orient='h', color='lightcoral')
plt.title("Boxplot de 'area_extra'  (valores <= Q3)")
plt.xlabel("area_extra")
plt.tight_layout()
plt.show()

# Boxplot horizontal
plt.figure(figsize=(10, 1.5))
sns.boxplot(data=data, orient='h', color='lightcoral')
plt.title("Boxplot de 'area_extra'")
plt.xlabel("area_extra")
plt.tight_layout()
plt.show()

Very extreme values are observed.

In [ ]:
# Original data including zeros
data = train_df['preco']

# Filter values above Q3 for the chart (keeping zeros)
q3 = data.quantile(0.75)
data_filtrada = data[data <= q3]

min_val = data_filtrada.min()
max_val = data_filtrada.max()
mean_val = data_filtrada.mean()
median_val = data_filtrada.median()
q1 = data_filtrada.quantile(0.25)

# Display the chart
plt.figure(figsize=(8, 5))
sns.histplot(data=data_filtrada, kde=True, bins=30, color='skyblue')
plt.title("Distribution of 'preco' (valores <= Q3)")
plt.xlabel('preco')
plt.ylabel("Frequency")

# Linhas de estatísticas
plt.axvline(min_val, color='red', linestyle='--', label=f'Mín: {min_val:.2f}')
plt.axvline(max_val, color='green', linestyle='--', label=f'Máx: {max_val:.2f}')
plt.axvline(mean_val, color='orange', linestyle='--', label=f'Média: {mean_val:.2f}')
plt.axvline(median_val, color='purple', linestyle='--', label=f'Mediana: {median_val:.2f}')
plt.axvline(q1, color='blue', linestyle='--', label=f'Q1: {q1:.2f}')
plt.axvline(q3, color='blue', linestyle=':', label=f'Q3: {q3:.2f}')

# Legenda incluindo a contagem de zeros
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Boxplot horizontal
plt.figure(figsize=(10, 1.5))
sns.boxplot(data=data_filtrada, orient='h', color='lightcoral')
plt.title("Boxplot de 'preco' (valores <= Q3)")
plt.xlabel("preco")
plt.tight_layout()
plt.show()

# Boxplot horizontal
plt.figure(figsize=(10, 1.5))
sns.boxplot(data=data, orient='h', color='lightcoral')
plt.title("Boxplot de 'preco'")
plt.xlabel("preco")
plt.tight_layout()
plt.show()

Again, extreme prices are observed, both very high and very low. They will need to be filtered, as this is the target variable.

## Processing the 'diferenciais' (Features) Column

### Analysis of Words Present in the Column

In [ ]:
# Collect words
todos_os_termos = []

# Basic list of stop words (stopwords)
stopwords = {
    'e', 'nenhum', 'de', 'frente', 'para', 'o', 'com'
}

for texto in train_df['diferenciais'].dropna():
    palavras = texto.lower().split()
    palavras_filtradas = [p for p in palavras if p not in stopwords]
    todos_os_termos.extend(palavras_filtradas)

# Contagem das frequências
contagem = Counter(todos_os_termos)

# Exibição das 35 palavras mais frequentes
print("Distribuição dos diferenciais:")
for termo, freq in contagem.most_common(35):
    print(f"{termo}: {freq}")


Let's count the number of 1s in the existing binary feature columns to compare the occurrence counts.

In [ ]:
colunas = [
    'churrasqueira', 'estacionamento', 'piscina', 'playground', 'quadra',
    's_festas', 's_jogos', 's_ginastica', 'sauna', 'vista_mar'
]

# Contagem dos 1 (presença do equipamento)
contagem = train_df[colunas].sum()

# Exibição
for col, val in contagem.items():
    print(f"{col}: {val}")


By comparing these two lists, we observe that the existing binary columns — such as churrasqueira, estacionamento, piscina, among others — have exactly the same number of 1 values as the number of times these words appear in the 'diferenciais' column. This shows that this information was already represented in a structured way in the data.

Below is the list of words obtained from the 'diferenciais' column. At the top are words whose features already exist and whose information we already have. At the bottom are features that do not yet exist and need to be created to increase the available information:

- churrasqueira *
- estacionamento *
- piscina *
- playground *
- quadra * (= soccer field + multipurpose court)
- s_festas *
- s_jogos *
- s_ginastica *
- sauna *
- vista_mar *

New features to create:

- futebol
- poliesportiva
- esquina
- copa
- children
- vestiário
- hidromassagem
- squash

### Word Extraction and New Feature Creation from 'diferenciais'

In [ ]:
def extrair_diferenciais(df, coluna='diferenciais'):
    # Lista de palavras-chave a extrair
    palavras_chave = [
        'futbol',
        'poliesportiva',
        'esquina',
        'copa',
        'children',
        'vestiario',
        'hidromassagem',
        'squash'
    ]
    
    # Ensure the column is of type string
    df[coluna] = df[coluna].astype(str).str.lower()
    
    # Create a column for each keyword
    for palavra in palavras_chave:
        df[palavra] = df[coluna].apply(lambda x: 1 if palavra in x.split() else 0)
    
    return df

extrair_diferenciais(train_df)
extrair_diferenciais(test_df)

## Feature Engineering

Creation of new relevant features from the information already available.

In [ ]:
def engenharia_de_features(df):
    df = df.copy()

    # Área total
    df['area_total'] = df['area_util'] + df['area_extra']

    # Área útil por quarto
    df['area_por_quarto'] = df['area_util'] / df['quartos'].replace(0, np.nan)

    # Área útil por cômodo (quartos + suítes)
    df['area_por_comodo'] = df['area_util'] / (df['quartos'] + df['suites']).replace(0, np.nan)

    # Razão suítes / quartos
    df['proporcao_suites'] = df['suites'] / df['quartos'].replace(0, np.nan)

    # Equipamentos de luxo
    equipamentos = [
        'churrasqueira', 'estacionamento', 'piscina', 'playground', 'quadra', 's_festas',
        's_jogos', 's_ginastica', 'sauna', 'vista_mar', 'futbol', 'poliesportiva',
        'esquina', 'copa', 'children', 'vestiario', 'hidromassagem', 'squash'
    ]
    df['n_equipamentos'] = df[equipamentos].sum(axis=1)

    return df

train_df = engenharia_de_features(train_df)
test_df = engenharia_de_features(test_df)

## Outlier Treatment

In [ ]:
colunas = ['area_util', 'preco']

IQR = 1.55  # Valor que obtém os melhores resultados após várias tentativas

for col in colunas:
    dados = train_df[col].dropna()
    q1 = dados.quantile(0.25)
    q3 = dados.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - IQR * iqr
    limite_superior = q3 + IQR * iqr
    valor_min = dados.min()
    valor_max = dados.max()

    print(f"\n--- {col} ---")
    print(f"Mínimo : {valor_min:.2f}")
    print(f"Máximo : {valor_max:.2f}")
    print(f"Q1 : {q1:.2f}")
    print(f"Q3 : {q3:.2f}")
    print(f"IQR : {iqr:.2f}")
    print(f"Limite inferior (Q1 - {IQR}*IQR) : {limite_inferior:.2f}")
    print(f"Limite superior (Q3 + {IQR}*IQR) : {limite_superior:.2f}")

# Tratamento especial para 'area_extra' (excluindo os zeros)
dados = train_df['area_extra']
dados_filtrados = dados[(dados != 0) & (~dados.isna())]

q1 = dados_filtrados.quantile(0.25)
q3 = dados_filtrados.quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - IQR * iqr
limite_superior = q3 + IQR * iqr
valor_min = dados_filtrados.min()
valor_max = dados_filtrados.max()

print(f"\n--- area_extra (sem zeros) ---")
print(f"Mínimo : {valor_min:.2f}")
print(f"Máximo : {valor_max:.2f}")
print(f"Q1 : {q1:.2f}")
print(f"Q3 : {q3:.2f}")
print(f"IQR : {iqr:.2f}")
print(f"Limite inferior (Q1 - {IQR}*IQR) : {limite_inferior:.2f}")
print(f"Limite superior (Q3 + {IQR}*IQR) : {limite_superior:.2f}")


In [ ]:
# --- Outlier removal for preco, area_util, area_extra using the IQR method ---

# --- Definir os limites mínimo e máximo ---
limites = {
    'area_util': (0, 2045.00),
    'preco': (1000, 1566250.00),  # Only column where we filter outliers
    'area_extra': (0, 17450.00)
}

# --- Aplicar o filtro segundo os limites definidos ---
mascaras = []

for coluna, (val_min, val_max) in limites.items():
    mascara = (train_df[coluna] >= val_min) & (train_df[coluna] <= val_max)
    mascaras.append(mascara)

# Combinar as máscaras
mascara_total = mascaras[0]
for m in mascaras[1:]:
    mascara_total &= m

# Filter the DataFrame
train_df_filtrado = train_df[mascara_total]

# --- Exibição dos resultados ---
print("Filtro baseado em limites manuais:")
for col, (vmin, vmax) in limites.items():
    print(f"- {col}: entre {vmin} e {vmax}")

print(f"\nLinhas antes do filtro: {len(train_df)}")
print(f"Linhas após o filtro: {len(train_df_filtrado)}")
print(f"Número de linhas removidas: {len(train_df) - len(train_df_filtrado)}")

## Feature Importance with LGBMClassifier

This code prepares the data for training a LightGBM model. It encodes categorical columns with LabelEncoder to convert categories into numbers. Then it separates the data into explanatory variables (X) and target variable (y).

The LGBMClassifier model is then trained on this data. After training, each variable's importance is extracted via feature_importances_. This way, we identify the most important features for model training, as well as those that can be removed.

In [ ]:
# Preparation
df_model = train_df_filtrado.copy()

# Simple encoding of categorical columns
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

# Separação X / y
X = df_model.drop(columns=['Id', 'preco'])
y = df_model['preco']

# Initial model
model = LGBMRegressor(
    force_row_wise=True,
    random_state=42,
    n_estimators=100,
    learning_rate=0.1
)
model.fit(X, y)

# Importância
importances = pd.Series(model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

# Seleção das variáveis
top = importances_sorted.head(40)

# Exibição das 10 variáveis mais importantes
plt.figure(figsize=(10, 5))
top.plot(kind='barh', color='green')
plt.title("Features by Decreasing Importance (LightGBM Regressor)")
plt.gca().invert_yaxis()
plt.xlim(0, top.max() + 20)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Selection by threshold ---
limite_importancia = 0
features_a_manter = importances_sorted[importances_sorted >= limite_importancia].index.tolist()

# Colunas a manter + o alvo + eventualmente id_solicitante
colunas_a_manter_treino = features_a_manter + ['Id', 'preco']
colunas_a_manter_teste = features_a_manter + ['Id']

# Remove unwanted columns
train_df_reduzido = train_df_filtrado[colunas_a_manter_treino].copy()
test_df_reduzido = test_df[colunas_a_manter_teste].copy()

num_variaveis_removidas = len(importances_sorted) - len(features_a_manter)
print(f"Número de variáveis mantidas: {len(features_a_manter)}")
print(f"Número de variáveis removidas: {num_variaveis_removidas}")


After several attempts, we observed that the regression model handles data redundancy (multicollinearity) well. Removing useless columns does not change the final result, so we chose not to remove any.

## Data Preprocessing

In [ ]:
# --- Preprocessing ---
def preprocess(df):
    df = df.copy()

    # Identificar automaticamente as colunas categóricas (do tipo 'object' ou 'category')
    colunas_categoricas = df.select_dtypes(include=['object', 'category']).columns.tolist()

    # Codificação OneHot
    df = pd.get_dummies(df, columns=colunas_categoricas, drop_first=True)
    return df

- We start by separating the explanatory variables X and the target y.

- No imputation is needed since there are no missing values, and no scaling is applied since it is unnecessary with LGBM; preprocessing is limited to one-hot encoding.

- For the test set, we apply the same one-hot encoder to maintain transformation consistency.

- Then we harmonize columns between train and test: we add missing columns in test (filled with zeros) and remove extra columns.

- Finally, we ensure that the column order is identical in both DataFrames.

In [ ]:
# Separate features and target
X = train_df_reduzido.drop(columns=['Id', 'preco'])
y = train_df_reduzido['preco']

# Training set preprocessing
X_processado = preprocess(X)

# Test set preprocessing
test_processado = preprocess(test_df_reduzido.drop(columns=['Id']))

# --- Train/test harmonization ---
# Add missing columns in test_processed with value 0
colunas_faltando = set(X_processado.columns) - set(test_processado.columns)
for col in colunas_faltando:
    test_processado[col] = 0

# Remove extra columns that exist in test_processed but not in X_processed
colunas_extras = set(test_processado.columns) - set(X_processado.columns)
test_processado.drop(columns=colunas_extras, inplace=True)

# Reordenar as colunas do test_processado para corresponder à ordem de X_processado
test_processado = test_processado[X_processado.columns]

This line splits the preprocessed data (X_processed and y) into two sets:

- A training set (X_train, y_train) containing 80% of the data,

- A validation set (X_val, y_val) with the remaining 20%.

Validation is used to evaluate the model's performance on data unseen during training.

The random_state=42 parameter fixes the random seed to ensure reproducibility.

In [ ]:
# --- Train / validation split ---
X_train, X_val, y_train, y_val = train_test_split(X_processado, y, test_size=0.2, random_state=42)

## Model Training and Cross-Validation

We start by defining the RMSPE evaluation function:

In [ ]:
def rmspe(y_true, y_pred):
    # Evitar divisão por zero (caso tenha algum y_true == 0)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Filter only values where y_true != 0
    mask = y_true != 0
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

This block trains three different models:

Random Forest Regressor,  
XGBoost Regressor,  
LightGBM Regressor

Each model is trained on the training data and then tested on the validation set.

The RMSPE score is computed to compare their performance on this validation.

In [ ]:
# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)
print("Random Forest:")
print(" RMSPE:", rmspe(y_val, y_pred_rf))

# --- XGBoost ---
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_val)
print("XGBoost:")
print(" RMSPE:", rmspe(y_val, y_pred_xgb))

# --- LightGBM ---
lgbm = LGBMRegressor(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_val)
print("LightGBM:")
print(" RMSPE:", rmspe(y_val, y_pred_lgbm))


In [ ]:
# Cross-validation with RMSPE
scores_rf = cross_val_score(rf, X_processado, y, cv=5,
    scoring=lambda est, X, y: -rmspe(y, est.predict(X)))
print("RMSPE médio validação cruzada (Random Forest):", -scores_rf.mean())

scores_xgb = cross_val_score(xgb, X_processado, y, cv=5,
    scoring=lambda est, X, y: -rmspe(y, est.predict(X)))
print("RMSPE médio validação cruzada (XGBoost):", -scores_xgb.mean())

scores_lgbm = cross_val_score(lgbm, X_processado, y, cv=5,
    scoring=lambda est, X, y: -rmspe(y, est.predict(X)))
print("RMSPE médio validação cruzada (LightGBM):", -scores_lgbm.mean())

## Optimized Hyperparameter Search

This code performs hyperparameter optimization for a LightGBM regression model using the Optuna library.

- The objective function defines the hyperparameters to test (number of leaves, max depth, learning rate, etc.) with possible ranges provided by Optuna.

- The model is evaluated by 5-fold cross-validation using the RMSPE (Root Mean Squared Percentage Error) as the performance metric.

In [ ]:
# --- Optuna objective function ---
def objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 120),
        'max_depth': trial.suggest_int('max_depth', 5, 25),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42,
        'verbosity': -1
    }

    model = LGBMRegressor(**params)
    
    # Utilização de validação cruzada com RMSPE
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_processado, y, cv=kf,
                             scoring=lambda est, X, y: -rmspe(y, est.predict(X)))
    
    return -np.mean(scores)  # Optuna minimiza, então invertemos o RMSPE

# --- Execução do estudo ---
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

# --- Melhores hiperparâmetros ---
print("Melhores hiperparâmetros:", study.best_params)
print("Score RMSPE (validação cruzada):", study.best_value)

## Prediction on the Test Set and Submission File Preparation

In [ ]:
# Train the model with the best parameters
final_lgbm = LGBMRegressor(**study.best_params)
final_lgbm.fit(X_processado, y)

In [ ]:
# --- Prediction on test set and submission preparation ---
y_test_pred = final_lgbm.predict(test_processado)
y_test_pred = final_lgbm.predict(test_processado)

# Create the submission DataFrame
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'preco': y_test_pred
})

# Salvar em CSV
submission.to_csv('submission.csv', index=False)
print("Arquivo de submissão salvo: submission.csv")

## Conclusion

The final score obtained via Kaggle submission was: **0.2527** (RMSPE)

## Key Results

| Metric | Value |
|--------|-------|
| **Best model** | LightGBM Regressor |
| **Kaggle score (RMSPE)** | 0.2527 |
| **Hyperparameter optimization** | Optuna (5-fold CV) |
| **Outlier treatment** | IQR method (factor 1.55) |
| **Key finding** | Feature engineering from the text 'diferenciais' column added valuable signal |

**Key insight:** The model handles multicollinearity well — removing redundant features did not improve performance. The most impactful steps were outlier treatment and extracting structured features from the free-text amenities column.

# Complete Source Code in a Single Cell

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMRegressor
import optuna

train_df = pd.read_csv('data/conjunto_de_treinamento.csv')
test_df = pd.read_csv('data/conjunto_de_teste.csv')

def extrair_diferenciais(df, coluna='diferenciais'):
    palavras_chave = [
        'futbol', 'poliesportiva', 'esquina', 'copa',
        'children', 'vestiario', 'hidromassagem', 'squash'
    ]
    df[coluna] = df[coluna].astype(str).str.lower()
    for palavra in palavras_chave:
        df[palavra] = df[coluna].apply(lambda x: 1 if palavra in x.split() else 0)
    return df

extrair_diferenciais(train_df)
extrair_diferenciais(test_df)

def engenharia_de_features(df):
    df = df.copy()
    df['area_total'] = df['area_util'] + df['area_extra']
    df['area_por_quarto'] = df['area_util'] / df['quartos'].replace(0, np.nan)
    df['area_por_comodo'] = df['area_util'] / (df['quartos'] + df['suites']).replace(0, np.nan)
    df['proporcao_suites'] = df['suites'] / df['quartos'].replace(0, np.nan)
    equipamentos = [
        'churrasqueira', 'estacionamento', 'piscina', 'playground', 'quadra',
        's_festas', 's_jogos', 's_ginastica', 'sauna', 'vista_mar', 'futbol',
        'poliesportiva', 'esquina', 'copa', 'children', 'vestiario',
        'hidromassagem', 'squash'
    ]
    df['n_equipamentos'] = df[equipamentos].sum(axis=1)
    return df

train_df = engenharia_de_features(train_df)
test_df = engenharia_de_features(test_df)

limites = {
    'area_util': (0, 2045.00),
    'preco': (1000, 1566250.00),
    'area_extra': (0, 17450.00)
}

mascara_total = np.ones(len(train_df), dtype=bool)
for coluna, (val_min, val_max) in limites.items():
    mascara_total &= (train_df[coluna] >= val_min) & (train_df[coluna] <= val_max)
train_df_filtrado = train_df[mascara_total]

df_model = train_df_filtrado.copy()
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

X = df_model.drop(columns=['Id', 'preco'])
y = df_model['preco']

modelo_inicial = LGBMRegressor(force_row_wise=True, random_state=42, n_estimators=100, learning_rate=0.1)
modelo_inicial.fit(X, y)
importances = pd.Series(modelo_inicial.feature_importances_, index=X.columns)
features_a_manter = importances[importances > 0].index.tolist()

colunas_treino = features_a_manter + ['Id', 'preco']
colunas_teste = features_a_manter + ['Id']
train_df_reduzido = train_df_filtrado[colunas_treino].copy()
test_df_reduzido = test_df[colunas_teste].copy()

def preprocess(df):
    colunas_categoricas = df.select_dtypes(include=['object', 'category']).columns
    return pd.get_dummies(df, columns=colunas_categoricas, drop_first=True)

X_processado = preprocess(train_df_reduzido.drop(columns=['Id', 'preco']))
test_processado = preprocess(test_df_reduzido.drop(columns=['Id']))

colunas_faltando = set(X_processado.columns) - set(test_processado.columns)
for col in colunas_faltando:
    test_processado[col] = 0
colunas_extras = set(test_processado.columns) - set(X_processado.columns)
test_processado.drop(columns=colunas_extras, inplace=True)
test_processado = test_processado[X_processado.columns]

def rmspe(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

def objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 120),
        'max_depth': trial.suggest_int('max_depth', 5, 25),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42,
        'verbosity': -1
    }
    model = LGBMRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_processado, y, cv=kf,
                             scoring=lambda est, X, y: -rmspe(y, est.predict(X)))
    return -np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

final_lgbm = LGBMRegressor(**study.best_params)
final_lgbm.fit(X_processado, y)
y_test_pred = final_lgbm.predict(test_processado)

submission = pd.DataFrame({'Id': test_df['Id'], 'preco': y_test_pred})
submission.to_csv('submission.csv', index=False)
